In [11]:
!pip install rouge_score
!pip install datasets
!pip install evaluate

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=90c2fede5288b0b2b2ce7ac093ba6435ce086aa6a6f8f5819f6c9fff9ef74218
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 4.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8

In [12]:
import pandas as pd
import re
import nltk
import string
import unicodedata
from sklearn.model_selection import train_test_split
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
import math
from rouge_score import rouge_scorer
import matplotlib.pyplot as plt
import numpy as np
import evaluate

In [13]:
import warnings

warnings.filterwarnings("ignore", message="To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).")

In [14]:
data = pd.read_csv("/kaggle/input/finalnlp-1/data_3.csv")
data.count

<bound method DataFrame.count of                                                        en  \
0                                      muiriel is 20 now.   
1                                 that was an evil bunny.   
2                                 i was in the mountains.   
3                                 is it a recent picture?   
4                                 is it a recent picture?   
...                                                   ...   
201654    i put it to you , is this eliminating poverty ?   
201655         well over 1,000 people use those toilets .   
201656                    and that , to me , is the key .   
201657                       people are not the problem .   
201658  none of those are limited by geography , by sk...   

                                                       vi  
0                           bây giờ muiriel được 20 tuổi.  
1                              đó là một con thỏ hung ác.  
2                                  tôi từ trên núi xuố

In [15]:
data.isna().sum()

en    0
vi    0
dtype: int64

In [16]:
df = data.copy()
# df = df[:500000]

train_data, temp_data = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_data, test_data = train_test_split(temp_data, test_size=0.25, random_state=42)

## Tokenizer BPE

In [ ]:
import sentencepiece as spm
import os
from typing import List

In [ ]:
# Write English and Vietnamese files
with open("BPE_200/train_en.en", 'w', encoding='utf-8') as f_en:
    for sentence in train_data['en']:
        f_en.write(sentence.strip() + '\n')

with open("BPE_200/train_vi.vi", 'w', encoding='utf-8') as f_vi:
    for sentence in train_data['vi']:
        f_vi.write(sentence.strip() + '\n')

# Create combined file for BPE training
with open("BPE_200/combined.txt", 'w', encoding='utf-8') as f_combined:
    for sentence in train_data['en']:
        f_combined.write(sentence.strip() + '\n')
    for sentence in train_data['vi']:
        f_combined.write(sentence.strip() + '\n')

In [ ]:
# Step 2: Train BPE Tokenizer
def train_bpe_tokenizer(input_file: str, model_prefix: str, vocab_size: int = 32000) -> None:
    """
    Train a SentencePiece BPE tokenizer on the input file.

    Args:
        input_file: Path to combined text file
        model_prefix: Prefix for output model files (.model and .vocab)
        vocab_size: Size of the vocabulary
    """
    spm.SentencePieceTrainer.Train(
        input=input_file,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        normalization_rule_name='nmt_nfkc',  # Chuẩn hóa Unicode
        character_coverage= 1,  # Cover nearly all characters (important for Vietnamese)
        model_type='bpe',           # Use BPE algorithm
        max_sentence_length=10000,  # Handle long sentences if any
        input_sentence_size=1600000,  # Number of sentences to sample
        shuffle_input_sentence=True,  # Shuffle for better training
        add_dummy_prefix=True,      # Thêm tiền tố để đảm bảo tất cả các từ được xem xét tương tự
        byte_fallback=False,         # Xử lý ký tự không đúng bằng byte fallback
        pad_id=0,                   # Padding token
        unk_id=1,                   # Unknown token
        bos_id=2,                   # Beginning of sentence
        eos_id=3,                   # End of sentence
        train_extremely_large_corpus=True,
        split_digits=True,          # Tách chữ số
        split_by_whitespace=True,   # Tách theo khoảng trắng đầu tiên
        split_by_number=True,       # Tách chữ và số
        treat_whitespace_as_suffix=False,  # Xử lý khoảng trắng như một tiền tố
        user_defined_symbols=['<s>', '</s>', ',', '.', '?', '!'] # Thêm các token đặc biệt
    )
    print(f"Trained BPE tokenizer: {model_prefix}.model and {model_prefix}.vocab")

In [ ]:
# Step 3: Test Tokenizer
def test_tokenizer(model_file: str, test_sentences: List[str]) -> None:
    """
    Test the tokenizer on sample sentences.

    Args:
        model_file: Path to trained .model file
        test_sentences: List of sentences to test
    """
    sp = spm.SentencePieceProcessor()
    sp.load(model_file)

    print("\nTesting tokenizer:")
    for sentence in test_sentences:
        tokens = sp.encode_as_pieces(sentence)
        token_ids = sp.encode_as_ids(sentence)
        decoded = sp.decode(token_ids)
        print(f"Sentence: {sentence}")
        print(f"Tokens: {tokens}")
        print(f"Token IDs: {token_ids}")
        print(f"Decoded: {decoded}")
        print(f"Number of tokens: {len(tokens)}\n")

In [ ]:
# OUTPUT_EN = "train.en"
# OUTPUT_VI = "train.vi"
COMBINED_FILE = "BPE_200/combined.txt"
MODEL_PREFIX = "BPE_200/bpe_tokenizer_40"
VOCAB_SIZE = 40000

print("\nStep 2: Training BPE tokenizer...")
train_bpe_tokenizer(COMBINED_FILE, MODEL_PREFIX, VOCAB_SIZE)

# Step 3: Test tokenizer
print("\nStep 3: Testing tokenizer...")
test_sentences = [
    "hello, how are you?",  # English
    "chào bạn, bạn khỏe không?",  # Vietnamese
    "i love to learn new things.",  # English
    "tôi thích học những điều mới."  # Vietnamese
]
test_tokenizer(f"{MODEL_PREFIX}.model", test_sentences)

print("Tokenizer training complete! Ready for next steps.")


In [ ]:
# with open("BPE_200/bpe_tokenizer.vocab", encoding='utf-8') as f:
#     vocab = [line.strip().split('\t')[0] for line in f]
# print("Số lượng token:", len(vocab))


## Transformer

In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    MBartForConditionalGeneration, MBart50TokenizerFast,
    T5ForConditionalGeneration, T5TokenizerFast,
    Trainer, TrainingArguments, TrainerCallback
)
import sentencepiece as spm
from typing import Dict, List, Tuple
import os
# import "/kaggle/input/finalnlp-1/TransformerModels.py" as tfm
# import TransformerScratch as tfs
# from tqdm import tqdm
from tqdm.notebook import tqdm
from transformers import MarianTokenizer, MarianMTModel

In [18]:
import sys
sys.path.append("/kaggle/input/finalnlp-1")  # Thêm thư mục chứa file vào đường dẫn tìm kiếm

import TransformerModels as tfm  # Nhập module


In [19]:
BPE_MODEL = "/kaggle/input/bpe_trans/pytorch/bpe_trans/1/bpe_tokenizer_20.model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
# Load BPE tokenizer for Transformer from scratch
sp = spm.SentencePieceProcessor()
sp.load(BPE_MODEL)

True

In [34]:
class TranslationDataset(Dataset):
    def __init__(self, data: pd.DataFrame, tokenizer, src_lang: str = "en", tgt_lang: str = "vi", max_length: int = 25, is_t5: bool = False, is_sentencepiece: bool = False):
        self.data = data
        self.tokenizer = tokenizer
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.max_length = max_length
        self.is_t5 = is_t5
        self.is_sentencepiece = is_sentencepiece

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        src_text = self.data.iloc[idx]['en']
        tgt_text = self.data.iloc[idx]['vi']

        # For T5, add task prefix
        if self.is_t5:
            src_text = f"translate English to Vietnamese: {src_text}"

        if self.is_sentencepiece:
            # Tokenize source and target using encode_as_ids
            src_token_ids = self.tokenizer.encode_as_ids(f"<SRC> {src_text} <TGT>")
            tgt_token_ids = self.tokenizer.encode_as_ids(tgt_text) + [self.tokenizer.piece_to_id("<EOS>")]

            # Padding and truncation
            src_token_ids = src_token_ids[:self.max_length]  # Truncate
            src_token_ids += [0] * (self.max_length - len(src_token_ids))  # Pad

            tgt_token_ids = tgt_token_ids[:self.max_length]  # Truncate
            tgt_token_ids += [0] * (self.max_length - len(tgt_token_ids))  # Pad

            # Create attention mask
            src_attention_mask = [1] * len(src_token_ids)

            # Convert to tensors
            src_token_ids = torch.tensor(src_token_ids)
            src_attention_mask = torch.tensor(src_attention_mask)
            tgt_token_ids = torch.tensor(tgt_token_ids)

            # Return data
            return {
                'input_ids': src_token_ids.clone().detach(),
                'attention_mask': src_attention_mask.clone().detach(),
                'labels': tgt_token_ids.clone().detach(),
                "src_text": src_text,
                "tgt_text": tgt_text
            }
        else:
            # For Hugging Face tokenizers (mBART, T5)
            src_encoding = self.tokenizer(
                src_text,
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            tgt_encoding = self.tokenizer(
                tgt_text,
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            return {
                'input_ids': src_encoding['input_ids'].squeeze(),
                'attention_mask': src_encoding['attention_mask'].squeeze(),
                'labels': tgt_encoding['input_ids'].squeeze(),
                "src_text": src_text,
                "tgt_text": tgt_text
            }

train_dataset = TranslationDataset(train_data, sp, is_sentencepiece=True)
val_dataset = TranslationDataset(val_data, sp, is_sentencepiece=True)
test_dataset = TranslationDataset(test_data, sp, is_sentencepiece=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

### Train từ đầu

In [35]:
def train_transformer(model, train_loader, val_loader, epochs, device = 'cuda'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

    # Initialize best_valid_loss to infinity
    best_valid_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        # Added tqdm for training loop
        train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for batch in train_progress:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Create target input (shifted right)
            tgt_input = labels[:, :-1]
            tgt_output = labels[:, 1:]

            # Create masks using new function
            tgt_mask, src_key_padding_mask, tgt_key_padding_mask = tfm.create_transformer_masks(
                input_ids, tgt_input, attention_mask, device
            )

            optimizer.zero_grad()
            output = model(
                input_ids,
                tgt_input,
                tgt_mask=tgt_mask,
                src_key_padding_mask=src_key_padding_mask,
                tgt_key_padding_mask=tgt_key_padding_mask
            )
            loss = criterion(output.view(-1, output.size(-1)), tgt_output.contiguous().view(-1))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_progress.set_postfix({"loss": loss.item()})

    # Validation
        model.eval()
        valid_loss = 0
        # Added tqdm for validation loop
        valid_progress = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]", leave=False)
        with torch.no_grad():
            for batch in valid_progress:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                tgt_input = labels[:, :-1]
                tgt_output = labels[:, 1:]

                # Create masks using new function
                tgt_mask, src_key_padding_mask, tgt_key_padding_mask = tfm.create_transformer_masks(
                    input_ids, tgt_input, attention_mask, device
                )

                output = model(
                    input_ids,
                    tgt_input,
                    tgt_mask=tgt_mask,
                    src_key_padding_mask=src_key_padding_mask,
                    tgt_key_padding_mask=tgt_key_padding_mask
                )
                loss = criterion(output.view(-1, output.size(-1)), tgt_output.contiguous().view(-1))
                valid_loss += loss.item()
                valid_progress.set_postfix({"loss": loss.item()})

        avg_valid_loss = valid_loss / len(val_loader)
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader):.4f}, Valid Loss: {valid_loss/len(val_loader):.4f}")

        # Save model if validation loss improves
        if avg_valid_loss < best_valid_loss:
            best_valid_loss = avg_valid_loss
            save_path = f"best_transformer.pt"
            torch.save(model.state_dict(), save_path)
            print(f"Saved model at {save_path}")

    return model

In [36]:
VOCAB_SIZE = 20000
EPOCHS = 15

print("Training Transformer from scratch...")
transformer_model = tfm.TransformerModel(vocab_size=VOCAB_SIZE)
transformer_model = train_transformer(transformer_model, train_loader, val_loader, epochs=EPOCHS, device=DEVICE)
# torch.save(transformer_model.state_dict(), "transformer_from_scratch_32.pt")
print("Transformer from scratch saved to transformer_from_scratch.pt")

Training Transformer from scratch...


Epoch 1/15 [Train]:   0%|          | 0/5042 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch 1/15 [Valid]:   0%|          | 0/946 [00:00<?, ?it/s]

Epoch 1/15, Train Loss: 3.8956, Valid Loss: 2.7270
Saved model at best_transformer.pt


...

Epoch 14/15 [Valid]:   0%|          | 0/946 [00:00<?, ?it/s]

Epoch 14/15, Train Loss: 0.8246, Valid Loss: 1.2461
Saved model at best_transformer.pt


Epoch 15/15 [Train]:   0%|          | 0/5042 [00:00<?, ?it/s]

Epoch 15/15 [Valid]:   0%|          | 0/946 [00:00<?, ?it/s]

Epoch 15/15, Train Loss: 0.7874, Valid Loss: 1.2409
Saved model at best_transformer.pt
Transformer from scratch saved to transformer_from_scratch.pt


### Pretrained

In [ ]:
class TqdmCallback(TrainerCallback):
    def __init__(self):
        super().__init__()
        self.train_pbar = None
        self.eval_pbar = None
        self.best_eval_loss = float('inf')  # Track best validation loss
        self.output_dir = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_pbar = tqdm(total=state.max_steps, desc="Training", leave=True)
        self.output_dir = args.output_dir  # Store output directory

    def on_step_end(self, args, state, control, **kwargs):
        self.train_pbar.update(1)
        if state.log_history:
            loss = state.log_history[-1].get('loss', None)
            if loss:
                self.train_pbar.set_postfix({"loss": loss})

    def on_evaluate(self, args, state, control, **kwargs):
        if self.eval_pbar is None:
            self.eval_pbar = tqdm(total=state.num_eval_steps, desc="Evaluating", leave=True)

    def on_eval_step_end(self, args, state, control, **kwargs):
        if self.eval_pbar:
            self.eval_pbar.update(1)
            if state.log_history:
                eval_loss = state.log_history[-1].get('eval_loss', None)
                if eval_loss:
                    self.eval_pbar.set_postfix({"eval_loss": eval_loss})

    def on_epoch_end(self, args, state, control, model, **kwargs):
        # Save model if eval loss improves
        if state.log_history:
            eval_loss = state.log_history[-1].get('eval_loss', float('inf'))
            if eval_loss < self.best_eval_loss:
                self.best_eval_loss = eval_loss
                checkpoint_dir = os.path.join(self.output_dir, f"checkpoint-epoch-{state.epoch}")
                model.save_pretrained(checkpoint_dir)
                kwargs['tokenizer'].save_pretrained(checkpoint_dir)
                print(f"Saved model at {checkpoint_dir} with eval loss: {eval_loss:.4f}")

    def on_train_end(self, args, state, control, **kwargs):
        if self.train_pbar:
            self.train_pbar.close()

    def on_evaluate_end(self, args, state, control, **kwargs):
        if self.eval_pbar:
            self.eval_pbar.close()
            self.eval_pbar = None

### mBart

In [ ]:
# def fine_tune_mbart(
#     train_dataset: TranslationDataset,
#     valid_dataset: TranslationDataset,
#     output_dir: str = "./BestModels/mbart_finetuned"
# ):
#     model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
#     tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50-many-to-many-mmt", src_lang="en_XX", tgt_lang="vi_VN")

#     training_args = TrainingArguments(
#         output_dir=output_dir,
#         num_train_epochs=3,
#         per_device_train_batch_size=16,
#         per_device_eval_batch_size=16,
#         evaluation_strategy="epoch",
#         save_strategy="epoch",
#         learning_rate=1e-5,
#         save_total_limit=2,
#         load_best_model_at_end=True,
#         metric_for_best_model="loss",
#         greater_is_better=False,
#         fp16=True
#     )

#     trainer = Trainer(
#         model=model,
#         args=training_args,
#         train_dataset=train_dataset,
#         eval_dataset=valid_dataset,
#         tokenizer=tokenizer,
#         callbacks=[TqdmCallback()]  # Added TqdmCallback for progress bar
#     )

#     trainer.train()
#     trainer.save_model(output_dir)
#     tokenizer.save_pretrained(output_dir)
#     print(f"mBART fine-tuned and saved to {output_dir}")

In [ ]:
# mbart_tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50-many-to-many-mmt", src_lang="en_XX", tgt_lang="vi_VN")
# # mbart_train_dataset, mbart_valid_dataset, _ = prepare_data(CSV_FILE, mbart_tokenizer)
# mbart_train_dataset = TranslationDataset(train_data, mbart_tokenizer, is_sentencepiece=False)
# mbart_valid_dataset = TranslationDataset(val_data, mbart_tokenizer, is_sentencepiece=False)

# print("Fine-tuning mBART...")
# fine_tune_mbart(mbart_train_dataset, mbart_valid_dataset)

### T5

In [ ]:
def fine_tune_t5(
    train_dataset: TranslationDataset,
    valid_dataset: TranslationDataset,
    output_dir: str = "t5_finetuned"
):
    model = T5ForConditionalGeneration.from_pretrained("google/mt5-small")
    tokenizer = T5TokenizerFast.from_pretrained("google/mt5-small")

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=4,
        per_device_train_batch_size=8,  # Có thể dùng 16 vì mô hình nhẹ
        per_device_eval_batch_size=8,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=1e-5,
        save_total_limit=2,
        load_best_model_at_end=False,
        metric_for_best_model="loss",
        greater_is_better=False,
        fp16=True,
        report_to="none",
        gradient_accumulation_steps=2,  # Để ổn định với batch size hiệu quả
        gradient_checkpointing=True
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        tokenizer=tokenizer
    )

    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"T5 fine-tuned and saved to {output_dir}")
    torch.save(model.state_dict(), "t5_finetuned.pt")

In [ ]:
t5_tokenizer = T5TokenizerFast.from_pretrained("google/mt5-small")
# t5_train_dataset, t5_valid_dataset, _ = prepare_data(CSV_FILE, t5_tokenizer, is_t5=True)
t5_train_dataset = TranslationDataset(train_data, t5_tokenizer, is_sentencepiece=False)
t5_valid_dataset = TranslationDataset(val_data, t5_tokenizer, is_sentencepiece=False)

print("Fine-tuning T5...")
fine_tune_t5(t5_train_dataset, t5_valid_dataset)

### MarianMT

In [ ]:
def fine_tune_marianmt(
    train_dataset: TranslationDataset,
    valid_dataset: TranslationDataset,
    output_dir: str = "marianmt_finetuned"
):
    model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-vi")
    tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-vi")

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=5,
        per_device_train_batch_size=16,  # Có thể dùng 16 vì mô hình nhẹ
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=1e-5,
        save_total_limit=2,
        load_best_model_at_end=False,
        metric_for_best_model="loss",
        greater_is_better=False,
        fp16=True,
        report_to="none",
        gradient_accumulation_steps=2,  # Để ổn định với batch size hiệu quả
        gradient_checkpointing=True
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        tokenizer=tokenizer
    )

    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"MarianMT fine-tuned and saved to {output_dir}")
    # Sau khi fine-tune
    torch.save(model.state_dict(), "marianmt_finetuned.pt")

In [ ]:
marian_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-vi")
# t5_train_dataset, t5_valid_dataset, _ = prepare_data(CSV_FILE, t5_tokenizer, is_t5=True)
marian_train_dataset = TranslationDataset(train_data, marian_tokenizer, is_sentencepiece=False)
marian_valid_dataset = TranslationDataset(val_data, marian_tokenizer, is_sentencepiece=False)

print("Fine-tuning MarianMT...")
fine_tune_marianmt(marian_train_dataset, marian_valid_dataset)

## Đánh giá

In [37]:
class EvalTranslationDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=25, is_pretrained=False):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_pretrained = is_pretrained
        self.src_token = "<SRC>"
        self.tgt_token = "<TGT>"
        self.eos_token = "<EOS>"

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_text = self.data.iloc[idx]["en"]
        tgt_text = self.data.iloc[idx]["vi"]

        # Input cho MarianMT (pretrained): chỉ src_text
        if self.is_pretrained:
            input_text = src_text  # Không thêm prompt hoặc [TGT]
        # Input cho GPT (non-pretrained): <SRC> src_text <TGT>
        else:
            input_text = f"{self.src_token} {src_text} {self.tgt_token}"

        # Token hóa
        if self.is_pretrained:
            encoded = self.tokenizer(
                input_text,
                max_length=self.max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids = encoded["input_ids"].squeeze()
            attention_mask = encoded["attention_mask"].squeeze()
        else:
            encoded = self.tokenizer.encode(input_text)
            if len(encoded) > self.max_length:
                encoded = encoded[:self.max_length - 1] + [self.tokenizer.piece_to_id(self.eos_token)]
            else:
                encoded += [0] * (self.max_length - len(encoded))
            input_ids = torch.tensor(encoded, dtype=torch.long)
            attention_mask = (input_ids != 0).bool()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "src_text": src_text,
            "tgt_text": tgt_text
        }

In [67]:
def greedy_decode_transformer(model, input_ids, attention_mask, max_length, device, pad_id=0, eos_id=3, tgt_id=5):
    """
    Greedy decoding for TransformerModel (encoder-decoder) with per-sample EOS stopping.

    Args:
        model: TransformerModel instance
        input_ids: Tensor of shape [batch_size, seq_len]
        attention_mask: Tensor of shape [batch_size, seq_len]
        max_length: Maximum length of generated sequence
        device: Device to run model
        pad_id: Padding token ID
        eos_id: End-of-sequence token ID
        tgt_id: Target token ID (<TGT>)

    Returns:
        Tensor of shape [batch_size, max_length] with generated sequences
    """
    model.eval()
    batch_size = input_ids.size(0)

    # Khởi tạo tgt_ids với <TGT>
    tgt_ids = torch.full((batch_size, 1), tgt_id, device=device, dtype=torch.long)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    with torch.no_grad():
        for step in range(max_length - 1):
            if finished.all():
                break

            # Tạo tgt_mask
            tgt_mask = model.transformer.generate_square_subsequent_mask(tgt_ids.size(1)).to(device)

            # Forward pass
            outputs = model(
                src=input_ids,
                tgt=tgt_ids,
                src_key_padding_mask=~attention_mask.bool(),
                tgt_mask=tgt_mask
            )
            logits = outputs[:, -1, :]  # Logits của token cuối
            next_token = torch.argmax(logits, dim=-1, keepdim=True)

            # Thêm token mới, chỉ cho mẫu chưa xong
            next_token = torch.where(finished.unsqueeze(-1), pad_id, next_token)
            tgt_ids = torch.cat([tgt_ids, next_token], dim=-1)

            # Cập nhật trạng thái hoàn thành
            finished |= next_token.squeeze(-1).eq(eos_id)

    # Padding nếu cần
    if tgt_ids.size(-1) < max_length:
        padding = torch.full((batch_size, max_length - tgt_ids.size(-1)), pad_id, device=device, dtype=torch.long)
        tgt_ids = torch.cat([tgt_ids, padding], dim=-1)
    else:
        tgt_ids = tgt_ids[:, :max_length]

    return tgt_ids

def beam_search_decode_transformer(model, input_ids, attention_mask, max_length, device, num_beams=3, pad_id=0, eos_id=3, tgt_id=5, length_penalty=0.6):
    """
    Beam search decoding for TransformerModel with per-sample EOS stopping and length penalty.

    Args:
        model: TransformerModel instance
        input_ids: Tensor of shape [batch_size, seq_len]
        attention_mask: Tensor of shape [batch_size, seq_len]
        max_length: Maximum length of generated sequence
        device: Device to run model
        num_beams: Number of beams
        pad_id: Padding token ID
        eos_id: End-of-sequence token ID
        tgt_id: Target token ID (<TGT>)
        length_penalty: Length penalty coefficient (higher favors shorter sequences)

    Returns:
        Tensor of shape [batch_size, max_length] with generated sequences
    """
    model.eval()
    batch_size = input_ids.size(0)

    # Khởi tạo beams [batch_size, num_beams, seq_len=1]
    beams = torch.full((batch_size, num_beams, 1), tgt_id, device=device, dtype=torch.long)
    beam_scores = torch.zeros((batch_size, num_beams), device=device)
    beam_scores[:, 1:] = -1e9  # Ưu tiên beam đầu tiên
    finished = torch.zeros((batch_size, num_beams), dtype=torch.bool, device=device)
    sample_finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    with torch.no_grad():
        for step in range(max_length - 1):
            if sample_finished.all():
                break

            # Flatten để xử lý batch [batch_size*num_beams, seq_len]
            flat_beams = beams.view(-1, beams.size(-1))
            flat_scores = beam_scores.view(-1)

            # Tạo mask cho toàn batch
            tgt_mask = model.transformer.generate_square_subsequent_mask(flat_beams.size(1)).to(device)

            # Chuẩn bị input_ids mở rộng
            expanded_input_ids = input_ids.unsqueeze(1).expand(-1, num_beams, -1).reshape(-1, input_ids.size(-1))
            expanded_attention_mask = attention_mask.unsqueeze(1).expand(-1, num_beams, -1).reshape(-1, attention_mask.size(-1))

            # Forward pass batch
            outputs = model(
                src=expanded_input_ids,
                tgt=flat_beams,
                src_key_padding_mask=~expanded_attention_mask.bool(),
                tgt_mask=tgt_mask
            )

            logits = outputs[:, -1, :]
            log_probs = torch.log_softmax(logits, dim=-1).view(batch_size, num_beams, -1)

            # Áp dụng length penalty
            length = flat_beams.size(-1) + 1
            length_penalty_factor = ((5 + length) / 6) ** length_penalty
            next_scores = beam_scores.unsqueeze(-1) + log_probs / length_penalty_factor

            # Chọn top-k beams mới
            next_scores, next_indices = next_scores.view(batch_size, -1).topk(num_beams, dim=-1)
            next_beam_indices = next_indices // log_probs.size(-1)  # Chỉ số beam
            next_token_indices = next_indices % log_probs.size(-1)   # Chỉ số token

            # Cập nhật beams
            new_beams = torch.zeros((batch_size, num_beams, beams.size(-1)+1), device=device, dtype=torch.long)
            for b in range(batch_size):
                if sample_finished[b]:
                    # Padding beam cũ để khớp chiều dài
                    padding = torch.full((num_beams, 1), pad_id, device=device, dtype=torch.long)
                    new_beams[b] = torch.cat([beams[b], padding], dim=-1)
                    continue
                for j in range(num_beams):
                    beam_idx = next_beam_indices[b, j].item()
                    token_idx = next_token_indices[b, j].item()

                    # Nối beam cũ với token mới
                    new_beams[b, j] = torch.cat([
                        beams[b, beam_idx],
                        torch.tensor([token_idx], device=device)
                    ])

                    # Đánh dấu beams đã hoàn thành
                    if token_idx == eos_id:
                        finished[b, j] = True

                # Nếu tất cả beams của mẫu hoàn thành, đánh dấu mẫu xong
                if finished[b].all():
                    sample_finished[b] = True

            beams = new_beams
            beam_scores = next_scores

    # Chọn beam tốt nhất
    best_indices = beam_scores.argmax(dim=-1)
    best_beams = beams[torch.arange(batch_size), best_indices]

    # Padding nếu cần
    if best_beams.size(-1) < max_length:
        padding = torch.full((batch_size, max_length - best_beams.size(-1)), pad_id, device=device, dtype=torch.long)
        best_beams = torch.cat([best_beams, padding], dim=-1)
    else:
        best_beams = best_beams[:, :max_length]

    return best_beams

def evaluate_single_model(model, tokenizer, test_data, device="cuda", batch_size=16, max_length=25, num_beams=3, is_pretrained=False, use_greedy=False, model_type="transformer"):
    """
    Evaluate a single model on test data.

    Args:
        model: Model to evaluate (TransformerModel or GPT)
        tokenizer: Tokenizer (SentencePiece or HuggingFace)
        test_data: Test dataset (dict with 'en' and 'vi')
        device: Device to run model
        batch_size: Batch size
        max_length: Maximum length of generated sequence
        num_beams: Number of beams for beam search
        is_pretrained: Whether model is pretrained (e.g., MarianMT/T5)
        use_greedy: Use greedy decoding (True) or beam search (False)
        model_type: "transformer" or "gpt"

    Returns:
        Dict with BLEU, ROUGE-1, ROUGE-2, ROUGE-L scores
    """
    model = model.to(device)
    model.eval()

    test_dataset = EvalTranslationDataset(test_data, tokenizer, max_length=max_length, is_pretrained=is_pretrained)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    bleu_metric = evaluate.load("bleu")
    rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    predictions = []
    references = []
    progress_bar = tqdm(test_loader, desc="Evaluating", leave=False)

    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            src_texts = batch["src_text"]
            tgt_texts = batch["tgt_text"]

            if is_pretrained:
                outputs = model.generate(input_ids=input_ids,attention_mask=attention_mask,max_length=max_length,num_beams=1 if use_greedy else num_beams,early_stopping=True,pad_token_id=tokenizer.pad_token_id if hasattr(tokenizer, 'pad_token_id') else 0)
            else:
                pad_id = tokenizer.pad_id() if hasattr(tokenizer, 'pad_id') else 0
                eos_id = tokenizer.piece_to_id("<EOS>")
                if model_type == "transformer":
                    tgt_id = tokenizer.piece_to_id("<TGT>")
                    if use_greedy:
                        outputs = greedy_decode_transformer(model,input_ids,attention_mask,max_length, device,pad_id=pad_id,eos_id=eos_id)
                    else:
                        outputs = beam_search_decode_transformer(model,input_ids,attention_mask,max_length,device,num_beams=num_beams,pad_id=pad_id,eos_id=eos_id)
                    
                elif model_type == "gpt":
                    if use_greedy:
                        outputs = greedy_decode_gpt(model, input_ids,attention_mask, max_length,device,pad_id=pad_id,eos_id=eos_id)
                    else:
                        outputs = beam_search_decode_gpt(model,input_ids,attention_mask,max_length,device,num_beams=num_beams,pad_id=pad_id,eos_id=eos_id)
                else:
                    raise ValueError("model_type must be 'transformer' or 'gpt'")

            for i, output in enumerate(outputs):
                if is_pretrained:
                    pred_text = tokenizer.decode(output, skip_special_tokens=True)
                    pred_text = pred_text.split("[TGT]")[-1].strip() if "[TGT]" in pred_text else pred_text
                else:
                    token_ids = output.tolist()
                    pred_text = tokenizer.decode(token_ids)
                    # print(pred_text)
                    if "<TGT>" in pred_text:
                        # print(pred_text)
                        pred_text = pred_text.split("<TGT>")[-1].strip()
                    else:
                        pred_text = ""
                    for token in ["<SRC>", "<TGT>", "<EOS>", "<PAD>", "<s>", "</s>"]:
                        pred_text = pred_text.replace(token, "")
                    pred_text = re.sub(r'\.+[i\.s]+|\.+', '.', pred_text).strip()
                    if not pred_text:
                        pred_text = " "
                predictions.append(pred_text)
                references.append([tgt_texts[i]])

    # Debug: In mẫu predictions và references
    print("Sample predictions:", predictions[:3])
    print("Sample references:", references[:3])

    # Tính BLEU
    bleu_score = bleu_metric.compute(predictions=predictions, references=references)["bleu"]

    # Tính ROUGE
    rouge_scores = {"rouge1": 0, "rouge2": 0, "rougeL": 0}
    for pred, ref in zip(predictions, references):
        scores = rouge_scorer_obj.score(ref[0], pred)
        for key in rouge_scores:
            rouge_scores[key] += scores[key].fmeasure
    rouge_scores = {key: val / len(predictions) for key, val in rouge_scores.items()}

    torch.cuda.empty_cache()
    return {
        "bleu": bleu_score,
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"]
    }



In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# Đường dẫn đến thư mục chứa mô hình đã fine-tune
output_dir = "marianmt_finetuned"

# Load mô hình
model = MarianMTModel.from_pretrained(output_dir)

# Load tokenizer
tokenizer_marian = MarianTokenizer.from_pretrained(output_dir)

# Chuyển mô hình sang thiết bị (GPU/CPU)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(DEVICE)

evaluate_single_model(model, tokenizer=tokenizer_marian, test_data=test_data, is_pretrained=True)

In [68]:
VOCAB_SIZE = 20000
model = tfm.TransformerModel(vocab_size=VOCAB_SIZE)
model.load_state_dict(torch.load("/kaggle/working/best_transformer.pt"))
model = model.to(DEVICE)
# evaluate_single_model(model, tokenizer=sp, test_data = test_data)
evaluate_single_model(model, tokenizer=sp, test_data = test_data, use_greedy=True)

Evaluating:   0%|          | 0/631 [00:00<?, ?it/s]

Sample predictions: ['tom ném thư vào lò sưởi ấm.', 'tôi hy vọng họ thích tôi.', 'vậy tôi đã kết thúc ở những nơi như thế này . đây .']
Sample references: [['tom ném lá thư vào lò sưởi.'], ['tôi hy vọng họ thích tôi'], ['bởi vậy tôi đã dừng chân ở những nơi như thế này .']]


{'bleu': 0.2751313905924988,
 'rouge1': 0.6274712849838787,
 'rouge2': 0.4855137781061264,
 'rougeL': 0.5880166691348523}

In [ ]:
print(sp.piece_to_id("<SRC>"))
print(sp.piece_to_id("<TGT>"))
print(sp.piece_to_id("<EOS>"))
print(sp.piece_to_id("<PAD>"))
print(sp.piece_to_id("<unk>"))

In [ ]:
# Plot comparison
def plot_comparison(results, output_file="comparison_metrics.png"):
    models = results["model"]
    bleu_scores = results["bleu"]
    rouge1_scores = results["rouge1"]
    rouge2_scores = results["rouge2"]
    rougeL_scores = results["rougeL"]

    x = np.arange(len(models))
    width = 0.2

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - 1.5*width, bleu_scores, width, label="BLEU", color="skyblue")
    ax.bar(x - 0.5*width, rouge1_scores, width, label="ROUGE-1", color="lightgreen")
    ax.bar(x + 0.5*width, rouge2_scores, width, label="ROUGE-2", color="salmon")
    ax.bar(x + 1.5*width, rougeL_scores, width, label="ROUGE-L", color="lightcoral")

    ax.set_xlabel("Models")
    ax.set_ylabel("Scores")
    ax.set_title("BLEU and ROUGE Scores Comparison")
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=45, ha="right")
    ax.legend()

    plt.tight_layout()
    plt.savefig(output_file)
    plt.close()
    print(f"Comparison plot saved to {output_file}")

In [ ]:
models_config = {
        "Transformer": {
            "model_class": tfm.TransformerModel,
            "model_path": "./BestModels/transformer_from_scratch_32.pt",
            "model_args": {"vocab_size": VOCAB_SIZE},
            "tokenizer": sp,
            "is_pretrained": False
        },
        "GPT": {
            "model_class": gs.GPTModel,
            "model_path": "./BestModels/gpt_finetuned.pt",
            "model_args": {"vocab_size": VOCAB_SIZE, "d_model": 256, "nhead": 8, "num_layers": 6, "dim_feedforward": 1024, "max_length": MAX_LENGTH},
            "tokenizer": sp,
            "is_pretrained": False
        },
        # "mBART": {
        #     "model_class": MBartForConditionalGeneration,
        #     "model_path": "./BestModels/mbart_finetuned.pt",
        #     "tokenizer": MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50-many-to-many-mmt", src_lang="en_XX", tgt_lang="vi_VN"),
        #     "is_pretrained": True
        # },
        "T5": {
            "model_class": T5ForConditionalGeneration,
            "model_path": "./BestModels/t5_finetuned.pt",
            "tokenizer": T5TokenizerFast.from_pretrained("google/mt5-small"),
            "is_pretrained": True,
            "is_t5": True
        },
        "MarianMT": {
            "model_class": MarianMTModel,
            "model_path": "./BestModels//marianmt_finetuned.pt",
            "tokenizer": MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-vi"),
            "is_pretrained": True
        },
        "GPT2": {
            "model_class": GPT2LMHeadModel,
            "model_path": "./BestModels/gpt2_finetuned.pt",
            "tokenizer": GPT2TokenizerFast.from_pretrained("gpt2"),
            "is_pretrained": True
        },
        "GPT-Neo": {
            "model_class": GPTNeoForCausalModeling,
            "model_path": "./BestModels/EleutherAI_gpt-neo-125M_finetuned.pt",
            "tokenizer": AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M"),
            "is_pretrained": True
        }
        # "BLOOM": {
        #     "model_class": BloomForCausalModeling,
        #     "model_path": "./BestModels/bigscience_bloom-560m_finetuned.pt",
        #     "tokenizer": AutoTokenizer.from_pretrained("bigscience/bloom-560m"),
        #     "is_pretrained": True
        # }
    }

# Update tokenizer pad_token
for name, config in models_config.items():
    if config["is_pretrained"] and "pad_token" not in config["tokenizer"]:
        config["tokenizer"].pad_token = config["tokenizer"].eos_token

# Create test dataset
test_datasets = {}
for model_name, config in models_config.items():
    test_datasets[model_name] = TranslationDataset(
        test_data,
        config["tokenizer"],
        max_length=MAX_LENGTH,
        is_pretrained=config["is_pretrained"],
        is_t5=config.get("is_t5", False),
        is_sentencepiece=not config["is_pretrained"]
    )

# Evaluate models
results = {}
for model_name in models_config:
    results[model_name] = evaluate_models(test_datasets[model_name], {model_name: models_config[model_name]}, device=DEVICE)

# Combine results
combined_results = {
    "model": [],
    "bleu": [],
    "rouge1": [],
    "rouge2": [],
    "rougeL": []
}
for model_name in results:
    combined_results["model"].append(model_name)
    combined_results["bleu"].append(results[model_name]["bleu"][0])
    combined_results["rouge1"].append(results[model_name]["rouge1"][0])
    combined_results["rouge2"].append(results[model_name]["rouge2"][0])
    combined_results["rougeL"].append(results[model_name]["rougeL"][0])

# Plot comparison
plot_comparison(combined_results)